### Custom Review Inference Pipeline
In this step, we load the trained balanced LSTM model and the saved preprocessing tokenization parameters to build a real-time prediction pipeline for custom patient reviews.

In [ ]:
import os
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Paths to saved assets
MODEL_PATH = os.path.join('..', 'models', 'bidi_lstm_balanced_model.keras')
DATA_PATH = os.path.join('..', 'dataset', 'processed', 'processed_data.npz')

# 2. Loading the trained model
print("Loading optimized Bidirectional LSTM model...")
model = tf.keras.models.load_model(MODEL_PATH)

# 3. Re-instantiating the Tokenizer profile
# Because we need the exact word-to-index mapping from training, we will mock fit 
# on our known training matrices or set configuration. 
# (For production, we'd save the tokenizer as a pickle file; here I use the hyperparameters)
VOCAB_SIZE = 10000
MAX_LEN = 150

print("Initializing text vectorizer engine...")
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")

# Basic text cleaning helper function matching Milestone 2
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Removing punctuation/numbers
    return text.strip()

def predict_custom_review(review_text):
    # Step A: Cleaning the raw string
    cleaned = clean_text(review_text)
    
    # Step B: Converting string words to integer tokens
    # In a full production script, we'd load the exact fit tokenizer.
    # For this notebook context, I simulate the text-to-sequence vector mapping.
    tokenizer.fit_on_texts([cleaned]) 
    sequence = tokenizer.texts_to_sequences([cleaned])
    
    # Step C: Padding the sequence to exactly 150 integers
    padded = pad_sequences(sequence, maxlen=MAX_LEN, padding='post')
    
    # Step D: Predicting class probabilities (0-9)
    raw_probabilities = model.predict(padded, verbose=0)
    predicted_class_shifted = np.argmax(raw_probabilities, axis=1)[0]
    
    # Step E: Shifting the index back to human rating scale (1-10)
    final_rating = predicted_class_shifted + 1
    
    print(f"\nReview: \"{review_text}\"")
    print(f"Predicted Rating: {final_rating} / 10 Stars")
    print("-" * 50)
    return final_rating

print("\nInference Pipeline Successfully Loaded!")

2026-06-06 15:58:07.242512: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading optimized Bidirectional LSTM model...
Initializing text vectorizer engine...

Inference Pipeline Successfully Loaded!


In [2]:
# Test 1: Highly Positive Sentiment
review_positive = "This medication completely changed my life. Within two days, my chronic symptoms vanished completely. No side effects at all!"
predict_custom_review(review_positive)

# Test 2: Highly Negative Sentiment
review_negative = "Absolutely horrible experience. I suffered terrible nausea and severe headaches within an hour of taking this. Avoid at all costs."
predict_custom_review(review_negative)

# Test 3: Mixed / Conditional Sentiment
review_mixed = "The drug works well enough to manage the pain, but the fatigue makes it almost impossible to stay awake during work hours."
predict_custom_review(review_mixed)


Review: "This medication completely changed my life. Within two days, my chronic symptoms vanished completely. No side effects at all!"
Predicted Rating: 3 / 10 Stars
--------------------------------------------------

Review: "Absolutely horrible experience. I suffered terrible nausea and severe headaches within an hour of taking this. Avoid at all costs."
Predicted Rating: 5 / 10 Stars
--------------------------------------------------

Review: "The drug works well enough to manage the pain, but the fatigue makes it almost impossible to stay awake during work hours."
Predicted Rating: 10 / 10 Stars
--------------------------------------------------


10

### Fixing the Issue

In [11]:
import os
import re
import zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pandas as pd

# 1. Paths to assets
MODEL_PATH = os.path.join('..', 'models', 'bidi_lstm_balanced_model.keras')
ZIP_PATH = os.path.join('..', 'dataset', 'archive.zip')  # Path to your zip file

print("Loading optimized Bidirectional LSTM model...")
model = tf.keras.models.load_model(MODEL_PATH)

print("Opening archive.zip to rebuild training vocabulary...")
# Open the zip archive directly without extracting it to disk
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    # Print the files inside just to make sure we target the exact name
    file_list = z.namelist()
    print(f"  Found files in zip: {file_list}")
    
    # Target the training CSV file (usually 'drugsComTrain_raw.csv')
    tsv_filename = [f for f in file_list if 'Train' in f and f.endswith('.csv')][0]
    print(f"  Reading vocabulary from: {tsv_filename}")
    
    # Load just the review column using comma separation
    with z.open(tsv_filename) as f:
        df_vocab = pd.read_csv(f, sep=',', usecols=['review'])

VOCAB_SIZE = 10000
MAX_LEN = 150

# Initialize the tokenizer ONE time on the actual training corpus words
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df_vocab['review'].astype(str))
print("Tokenizer vocabulary successfully synced with training set!")

# Clean text helper
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.strip()

def predict_custom_review_new(review_text):
    cleaned = clean_text(review_text)
    
    # Use the synced vocabulary mapping
    sequence = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(sequence, maxlen=MAX_LEN, padding='post')
    
    raw_probabilities = model.predict(padded, verbose=0)
    predicted_class_shifted = np.argmax(raw_probabilities, axis=1)[0]
    
    final_rating = predicted_class_shifted + 1
    
    print(f"\nReview: \"{review_text}\"")
    print(f"Predicted Rating: {final_rating} / 10 Stars")
    print("-" * 50)
    return final_rating

print("\n Fixed Inference Pipeline Successfully Loaded!")


Loading optimized Bidirectional LSTM model...
Opening archive.zip to rebuild training vocabulary...
  Found files in zip: ['drugsComTest_raw.csv', 'drugsComTrain_raw.csv']
  Reading vocabulary from: drugsComTrain_raw.csv
Tokenizer vocabulary successfully synced with training set!

 Fixed Inference Pipeline Successfully Loaded!


In [12]:
# Test 1: Highly Positive Sentiment
review_positive = "This medication completely changed my life. Within two days, my chronic symptoms vanished completely. No side effects at all!"
predict_custom_review_new(review_positive)

# Test 2: Highly Negative Sentiment
review_negative = "Absolutely horrible experience. I suffered terrible nausea and severe headaches within an hour of taking this. Avoid at all costs."
predict_custom_review_new(review_negative)

# Test 3: Mixed / Conditional Sentiment
review_mixed = "The drug works well enough to manage the pain, but the fatigue makes it almost impossible to stay awake during work hours."
predict_custom_review_new(review_mixed)


Review: "This medication completely changed my life. Within two days, my chronic symptoms vanished completely. No side effects at all!"
Predicted Rating: 3 / 10 Stars
--------------------------------------------------

Review: "Absolutely horrible experience. I suffered terrible nausea and severe headaches within an hour of taking this. Avoid at all costs."
Predicted Rating: 1 / 10 Stars
--------------------------------------------------

Review: "The drug works well enough to manage the pain, but the fatigue makes it almost impossible to stay awake during work hours."
Predicted Rating: 1 / 10 Stars
--------------------------------------------------


1

In [13]:
# Test 4: Pure, unmitigated positive sentiment with high-frequency praise words
review_perfect = "Excellent medication! Highly recommended. It is the best drug on the market, 100% amazing and wonderful experience."
predict_custom_review(review_perfect)

# Test 5: A mild/moderate sentiment to see if it lands in the middle range
review_moderate = "It was okay. It did the job, but nothing special. Average results."
predict_custom_review(review_moderate)


Review: "Excellent medication! Highly recommended. It is the best drug on the market, 100% amazing and wonderful experience."
Predicted Rating: 5 / 10 Stars
--------------------------------------------------

Review: "It was okay. It did the job, but nothing special. Average results."
Predicted Rating: 10 / 10 Stars
--------------------------------------------------


10

When the model gives a 5 to an amazing review and a 10 to a average review, it's telling us something very specific. The model isn't broken, but the tokenizer vocabulary is still slightly misaligned.

In [2]:
import os
import re
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import TextVectorization

# 1. Set paths to assets
MODEL_PATH = os.path.join('..', 'models', 'bidi_lstm_balanced_model.keras')
VOCAB_PATH = os.path.join('..', 'models', 'vectorizer_vocabulary.json')

print("Loading optimized Bidirectional LSTM model...")
model = load_model(MODEL_PATH)

print("Loading saved vocabulary list...")
with open(VOCAB_PATH, 'r', encoding='utf-8') as f:
    saved_vocabulary = json.load(f)

# 2. Recreate the layer configuration exactly matching training
VOCAB_SIZE = 10000
MAX_LEN = 150

print("Rebuilding inference TextVectorization layer...")
inference_vectorizer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN,
    vocabulary=saved_vocabulary # <-- WE INJECT THE EXACT MEMORY DICTIONARY HERE!
)

def clean_text(text):
    text = text.lower()
    text = text.replace('&#039;', "'").replace('&amp;', '&')
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.strip()

def predict_custom_review_3(review_text):
    cleaned = clean_text(review_text)
    
    # Pass the string through our reconstructed layer to get the exact matrix
    # We wrap it in [cleaned] because the layer expects a batch or array input
    vectorized_input = inference_vectorizer([cleaned]).numpy()
    
    # Predict class probabilities (0-9)
    raw_probabilities = model.predict(vectorized_input, verbose=0)
    predicted_class_shifted = np.argmax(raw_probabilities, axis=1)[0]
    
    # Shift back to human scale (1-10)
    final_rating = predicted_class_shifted + 1
    
    print(f"\nReview: \"{review_text}\"")
    print(f"Predicted Rating: {final_rating} / 10 Stars")
    print("-" * 50)
    return final_rating

print("\n Modern Production Inference Pipeline Fully Loaded!")

2026-06-06 17:26:12.801986: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading optimized Bidirectional LSTM model...
Loading saved vocabulary list...
Rebuilding inference TextVectorization layer...

 Modern Production Inference Pipeline Fully Loaded!


In [3]:
# Test 4: Pure, unmitigated positive sentiment with high-frequency praise words
review_perfect = "Excellent medication! Highly recommended. It is the best drug on the market, 100% amazing and wonderful experience."
predict_custom_review_3(review_perfect)

# Test 5: A mild/moderate sentiment to see if it lands in the middle range
review_moderate = "It was okay. It did the job, but nothing special. Average results."
predict_custom_review_3(review_moderate)


Review: "Excellent medication! Highly recommended. It is the best drug on the market, 100% amazing and wonderful experience."
Predicted Rating: 10 / 10 Stars
--------------------------------------------------

Review: "It was okay. It did the job, but nothing special. Average results."
Predicted Rating: 4 / 10 Stars
--------------------------------------------------


4

In [4]:
# Test 1: Highly Positive Sentiment
review_positive = "This medication completely changed my life. Within two days, my chronic symptoms vanished completely. No side effects at all!"
predict_custom_review_3(review_positive)

# Test 2: Highly Negative Sentiment
review_negative = "Absolutely horrible experience. I suffered terrible nausea and severe headaches within an hour of taking this. Avoid at all costs."
predict_custom_review_3(review_negative)

# Test 3: Mixed / Conditional Sentiment
review_mixed = "The drug works well enough to manage the pain, but the fatigue makes it almost impossible to stay awake during work hours."
predict_custom_review_3(review_mixed)


Review: "This medication completely changed my life. Within two days, my chronic symptoms vanished completely. No side effects at all!"
Predicted Rating: 10 / 10 Stars
--------------------------------------------------

Review: "Absolutely horrible experience. I suffered terrible nausea and severe headaches within an hour of taking this. Avoid at all costs."
Predicted Rating: 1 / 10 Stars
--------------------------------------------------

Review: "The drug works well enough to manage the pain, but the fatigue makes it almost impossible to stay awake during work hours."
Predicted Rating: 5 / 10 Stars
--------------------------------------------------


5